# Regressão Linear com PyTorch — King County House Sales

## Objetivos

Este notebook adapta a estrutura do `2.6-LinearRegressionSalaryData.ipynb` para
o dataset **King County House Sales** (Kaggle).

Diferente do notebook de salários (1 feature → 1 saída), aqui temos:
- **15 features** (área, quartos, banheiros, localização, qualidade, etc.)
- **1 saída**: preço da casa em dólares
- **Regressão Linear Múltipla** com `nn.Linear(15, 1)`
- Divisão treino/validação (80/20)
- Métricas em escala real (RMSE e MAE em dólares)

## Importação dos pacotes

In [ ]:
%matplotlib inline
import torch
from torch import nn, optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(1234)
np.random.seed(42)

## Dataset: King County House Sales

**Fonte:** https://www.kaggle.com/datasets/harlfoxem/housesalesprediction

O dataset contém 21.613 vendas de casas no Condado de King (Seattle, EUA) entre
2014-2015. A célula abaixo gera um dataset sintético com as mesmas distribuições
estatísticas do original para fins didáticos.

**Principais features:**
| Feature | Descrição |
|---|---|
| `sqft_living` | Área interna (pés²) |
| `bedrooms` | Número de quartos |
| `bathrooms` | Número de banheiros |
| `grade` | Qualidade de construção (1-13) |
| `waterfront` | Vista para o mar (0/1) |
| `lat` / `long` | Localização geográfica |
| `yr_built` | Ano de construção |

In [ ]:
# Gera dataset sintético baseado nas distribuições reais do King County House Sales
# Para usar o CSV original do Kaggle, substitua esta célula por:
# df = pd.read_csv('kc_house_data.csv')

n = 21613
sqft_living   = np.random.lognormal(7.5, 0.35, n).clip(300, 13540).astype(int)
bedrooms      = np.random.choice([1,2,3,4,5,6,7,8], n, p=[0.02,0.12,0.35,0.30,0.13,0.05,0.02,0.01])
bathrooms     = np.random.choice([1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0], n,
                    p=[0.10,0.05,0.25,0.20,0.15,0.10,0.08,0.04,0.03])
floors        = np.random.choice([1,1.5,2,2.5,3], n, p=[0.45,0.08,0.38,0.04,0.05])
waterfront    = np.random.binomial(1, 0.007, n)
view          = np.random.choice([0,1,2,3,4], n, p=[0.85,0.05,0.05,0.025,0.025])
condition     = np.random.choice([1,2,3,4,5], n, p=[0.01,0.03,0.65,0.25,0.06])
grade         = np.random.choice(range(3,13), n,
                    p=[0.002,0.01,0.05,0.15,0.30,0.25,0.15,0.06,0.02,0.008])
yr_built      = np.random.randint(1900, 2016, n)
lat           = np.random.uniform(47.15, 47.78, n)
long_         = np.random.uniform(-122.52, -121.31, n)
sqft_lot      = np.random.lognormal(8.8, 1.0, n).clip(520, 1651359).astype(int)
sqft_above    = (sqft_living * np.random.uniform(0.5, 1.0, n)).astype(int)
sqft_basement = (sqft_living - sqft_above).astype(int)
sqft_living15 = (sqft_living * np.random.uniform(0.8, 1.2, n)).clip(300, 6210).astype(int)
sqft_lot15    = (sqft_lot    * np.random.uniform(0.7, 1.3, n)).clip(651, 871200).astype(int)

price = (50000 + sqft_living*130 + bedrooms*(-8000) + bathrooms*20000
         + floors*8000 + waterfront*350000 + view*35000 + grade*30000
         + (2016-yr_built)*(-400) + np.random.normal(0, 90000, n)
         ).clip(75000, 7700000).astype(int)

df = pd.DataFrame({
    'price': price, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
    'sqft_living': sqft_living, 'sqft_lot': sqft_lot, 'floors': floors,
    'waterfront': waterfront, 'view': view, 'condition': condition,
    'grade': grade, 'sqft_above': sqft_above, 'sqft_basement': sqft_basement,
    'yr_built': yr_built, 'lat': lat, 'long': long_,
    'sqft_living15': sqft_living15, 'sqft_lot15': sqft_lot15
})

print(f"Shape do dataset: {df.shape}")
print(f"\nEstatísticas do preço:")
print(df['price'].describe().apply(lambda x: f'${x:,.0f}'))

In [ ]:
# Visualização rápida: distribuição do preço e correlações com as principais features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição do preço
axes[0].hist(df['price']/1e6, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Preço ($ milhões)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição do Preço')

# Preço vs Área interna
axes[1].scatter(df['sqft_living'], df['price']/1e6, alpha=0.3, s=5, color='coral')
axes[1].set_xlabel('Área interna (sqft)')
axes[1].set_ylabel('Preço ($ milhões)')
axes[1].set_title('Preço vs Área Interna')

# Preço médio por grade
grade_price = df.groupby('grade')['price'].mean() / 1e6
axes[2].bar(grade_price.index, grade_price.values, color='mediumseagreen')
axes[2].set_xlabel('Grade (qualidade)')
axes[2].set_ylabel('Preço médio ($ milhões)')
axes[2].set_title('Preço Médio por Grade')

plt.tight_layout()
plt.show()

## Pré-processamento

### Por que StandardScaler?

Com 15 features em escalas muito diferentes (área: ~2000 sqft vs latitude: ~47),
o gradiente descendente sofreria fortemente sem normalização:
- Features com escala grande dominariam o gradiente
- Convergência extremamente lenta ou divergência
- `StandardScaler` transforma cada feature para média=0 e desvio=1

In [ ]:
# Seleção de features
FEATURES = [
    'sqft_living', 'bedrooms', 'bathrooms', 'floors', 'waterfront',
    'view', 'condition', 'grade', 'sqft_above', 'sqft_basement',
    'yr_built', 'lat', 'long', 'sqft_living15', 'sqft_lot15'
]
TARGET = 'price'

X = df[FEATURES].values.astype(np.float32)
y = df[[TARGET]].values.astype(np.float32)

# Normalização (StandardScaler: z = (x - μ) / σ)
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y)

# Divisão treino/validação (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

# Conversão para tensores PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)

print(f"Treino:    X={X_train_t.shape}  y={y_train_t.shape}")
print(f"Validação: X={X_val_t.shape}    y={y_val_t.shape}")
print(f"\nNúmero de features: {X_train_t.shape[1]}")

## Modelo da Rede

### Por que `nn.Linear(15, 1)`?

A regressão linear múltipla calcula:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_{15} x_{15} + b$$

O `nn.Linear(in_features=15, out_features=1)` encapsula exatamente isso:
- Inicializa automaticamente 15 pesos + 1 bias
- `bias=True` (padrão) — não precisamos da coluna de 1s manual

In [ ]:
n_features = X_train_t.shape[1]  # 15 features

# nn.Linear(15, 1) cria: 15 pesos + 1 bias = 16 parâmetros treináveis
model = torch.nn.Linear(n_features, 1)

# Contagem de parâmetros
total_params = sum(p.numel() for p in model.parameters())
print(f"Arquitetura: Linear({n_features} → 1)")
print(f"Parâmetros: {total_params} ({n_features} pesos + 1 bias)")
print(f"\nPesos iniciais:\n{model.weight.data}")
print(f"\nBias inicial: {model.bias.data}")

## Treinamento

### Função de perda e otimizador

- **`nn.MSELoss()`**: Mean Squared Error — penaliza erros grandes quadraticamente
- **`optim.SGD(lr=0.05)`**: Stochastic Gradient Descent com lr menor que o
  notebook anterior — com 15 features e escala maior, lr=0.3 poderia divergir

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)

### Funções de treino e validação por época

In [ ]:
def train_epoch(model, optimizer, criterion, X, y):
    """Executa um passo completo de treino: forward → loss → backward → step"""
    model.train()           # ativa dropout/batchnorm se houver (boa prática)
    optimizer.zero_grad()   # zera gradientes acumulados
    out  = model(X)
    loss = criterion(out, y)
    loss.backward()         # calcula gradientes automaticamente
    optimizer.step()        # w ← w - lr * grad
    return loss.item()

def validate_epoch(model, criterion, X, y):
    """Avalia o modelo sem atualizar pesos (modo inferência)"""
    model.eval()            # desativa dropout/batchnorm
    with torch.no_grad():   # desativa grafo computacional
        out  = model(X)
        loss = criterion(out, y)
    return loss.item()

### Laço de treinamento

In [ ]:
num_epochs   = 300
train_losses = []
val_losses   = []
w_grade_list = []   # evolução do peso da feature 'grade' (índice 7)
w_sqft_list  = []   # evolução do peso da feature 'sqft_living' (índice 0)

for epoch in range(num_epochs):
    tl = train_epoch(model, optimizer, criterion, X_train_t, y_train_t)
    vl = validate_epoch(model, criterion, X_val_t, y_val_t)
    train_losses.append(tl)
    val_losses.append(vl)

    # Rastreia dois pesos para visualização no espaço de parâmetros
    w_sqft_list.append(model.weight.data[0][0].item())   # sqft_living
    w_grade_list.append(model.weight.data[0][7].item())  # grade

    if (epoch + 1) % 50 == 0:
        print(f'Epoch[{epoch+1}/{num_epochs}]  '
              f'Train Loss: {tl:.4f}  |  Val Loss: {vl:.4f}')

## Avaliação

In [ ]:
# Curva de aprendizado
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss (MSE norm.)', color='steelblue')
ax.plot(val_losses,   label='Val Loss (MSE norm.)',   color='coral', linestyle='--')
ax.set_xlabel('Época')
ax.set_ylabel('MSE (escala normalizada)')
ax.set_title('Convergência do Treinamento — King County House Sales')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Métricas em escala real (dólares)
with torch.no_grad():
    y_pred_val  = model(X_val_t)
    mse_norm    = criterion(y_pred_val, y_val_t).item()

y_pred_real = y_scaler.inverse_transform(y_pred_val.numpy())
y_real      = y_scaler.inverse_transform(y_val_t.numpy())

rmse = np.sqrt(np.mean((y_pred_real - y_real)**2))
mae  = np.mean(np.abs(y_pred_real - y_real))
r2   = 1 - np.sum((y_real - y_pred_real)**2) / np.sum((y_real - y_real.mean())**2)

print("=" * 45)
print("         MÉTRICAS — CONJUNTO DE VALIDAÇÃO")
print("=" * 45)
print(f"  MSE (normalizado):    {mse_norm:.6f}")
print(f"  RMSE (em dólares):    $ {rmse:>12,.0f}")
print(f"  MAE  (em dólares):    $ {mae:>12,.0f}")
print(f"  R²:                   {r2:.4f}")
print("=" * 45)
print(f"\n→ O modelo erra em média $ {mae:,.0f} por casa")

In [ ]:
# Predito vs Real — validação
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: predito vs real
axes[0].scatter(y_real/1e6, y_pred_real/1e6, alpha=0.3, s=8, color='steelblue')
lims = [min(y_real.min(), y_pred_real.min())/1e6,
        max(y_real.max(), y_pred_real.max())/1e6]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Predição perfeita')
axes[0].set_xlabel('Preço Real ($ M)')
axes[0].set_ylabel('Preço Predito ($ M)')
axes[0].set_title('Predito vs Real (Validação)')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Distribuição dos resíduos
residuos = (y_pred_real - y_real).flatten()
axes[1].hist(residuos/1e3, bins=60, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('Resíduo ($ mil)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição dos Resíduos')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Importância dos pesos (peso absoluto normalizado)
pesos = model.weight.data.numpy().flatten()
bias  = model.bias.data.item()

peso_df = pd.DataFrame({
    'Feature': FEATURES,
    'Peso': pesos,
    'Peso Abs': np.abs(pesos)
}).sort_values('Peso Abs', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['coral' if w < 0 else 'steelblue' for w in peso_df['Peso']]
ax.barh(peso_df['Feature'], peso_df['Peso'], color=colors)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Peso (escala normalizada)')
ax.set_title('Pesos da Regressão Linear — Importância das Features')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(f"Bias: {bias:.4f}")
print("\nFeatures com maior impacto positivo no preço:")
top3 = peso_df.sort_values('Peso', ascending=False).head(3)
for _, row in top3.iterrows():
    print(f"  {row['Feature']:20s}  peso = {row['Peso']:+.4f}")
print("\nFeatures com maior impacto negativo:")
bot3 = peso_df.sort_values('Peso').head(3)
for _, row in bot3.iterrows():
    print(f"  {row['Feature']:20s}  peso = {row['Peso']:+.4f}")

In [ ]:
# Scatter plot: trajetória de dois pesos no espaço de parâmetros
# (sqft_living e grade — as duas features mais relevantes)
fig, ax = plt.subplots(figsize=(8, 6))

w0_old, w1_old = None, None
for w0, w1 in zip(w_sqft_list, w_grade_list):
    if w0_old is not None:
        ax.annotate('', xy=(w0, w1), xytext=(w0_old, w1_old),
                    arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.0))
    w0_old, w1_old = w0, w1

sc = ax.scatter(w_sqft_list, w_grade_list, c=range(num_epochs),
                cmap='Blues', s=25, zorder=5)
plt.colorbar(sc, ax=ax, label='Época')

ax.scatter(w_sqft_list[0],  w_grade_list[0],  color='green', s=150,
           zorder=6, marker='o', label=f'Início')
ax.scatter(w_sqft_list[-1], w_grade_list[-1], color='navy',  s=150,
           zorder=6, marker='s', label=f'Fim (época {num_epochs})')

ax.set_xlabel('w(sqft_living)', fontsize=12)
ax.set_ylabel('w(grade)',        fontsize=12)
ax.set_title('Trajetória de Dois Pesos durante o SGD', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()